# Norm Consolidation EDA

Quality audit of the embedding-cluster-merge consolidation pipeline.
Key questions:
1. Are clusters semantically coherent? (same norm, different wording)
2. Are there bad merges? (distinct norms lumped together)
3. Are the mega-clusters (>10 members) meaningful or noise?
4. Did the LLM merger produce faithful canonical articulations?
5. How much information-flow-relevant content survived consolidation?

In [12]:
import pandas as pd
import json
import numpy as np
from collections import Counter
from IPython.display import display, HTML

consolidated = pd.read_parquet(
    '/share/pierson/matt/UAIR/outputs/2026-03-17/09-54-08/norm_consolidation_from_clusters/outputs/consolidation/consolidated_norms.parquet'
)
raw_norms = pd.read_parquet('/share/pierson/matt/n2s4cir/data/fiction10/structured_norms.parquet')

ID_TO_TITLE = {
    '1984': '1984', '541': 'Age of Innocence', '11': 'Alice', '1399': 'Anna Karenina',
    '1023': 'Bleak House', '1184': 'Count of Monte Cristo', '135': 'Les Mis',
    '145': 'Middlemarch', '4078': 'Dorian Gray', '1342': 'Pride & Prejudice',
}
consolidated['title'] = consolidated['group_key'].map(ID_TO_TITLE)

multi = consolidated[consolidated['cluster_size'] > 1].copy()
singletons = consolidated[consolidated['cluster_size'] == 1].copy()

print(f'Total canonical norms: {len(consolidated):,}')
print(f'  Singletons (pass-through): {len(singletons):,} ({len(singletons)/len(consolidated)*100:.1f}%)')
print(f'  Merged clusters: {len(multi):,} ({len(multi)/len(consolidated)*100:.1f}%)')
print(f'  Source norms merged: {multi["cluster_size"].sum():,}')

Total canonical norms: 8,534
  Singletons (pass-through): 7,789 (91.3%)
  Merged clusters: 745 (8.7%)
  Source norms merged: 8,093


## 1. Cluster Size Distribution

In [ ]:
print('Cluster size distribution (merged only):')
print(multi['cluster_size'].describe())
print()
print('Cluster size histogram:')
bins = [2, 3, 5, 10, 20, 50, 100, 1000]ß
labels = ['2', '3-4', '5-9', '10-19', '20-49', '50-99', '100+']
multi['size_bin'] = pd.cut(multi['cluster_size'], bins=bins, labels=labels, right=False)
print(multi['size_bin'].value_counts().sort_index())
print()
print('Mega-clusters (>20 members):')
mega = multi[multi['cluster_size'] > 20].sort_values('cluster_size', ascending=False)
for _, r in mega.iterrows():
    print(f'  [{r["title"]}] n={r["cluster_size"]}: {r["canonical_norm_articulation"]}')

Cluster size distribution (merged only):
count    745.000000
mean      10.863087
std       53.790364
min        3.000000
25%        3.000000
50%        4.000000
75%        6.000000
max      807.000000
Name: cluster_size, dtype: float64

Cluster size histogram:
size_bin
2          0
3-4      425
5-9      226
10-19     68
20-49     12
50-99      3
100+      11
Name: count, dtype: int64

Mega-clusters (>20 members):
  [Middlemarch] n=807: Individuals should engage in actions that align with their roles and responsibilities when they have the ability or opportunity to do so.
  [Count of Monte Cristo] n=805: Individuals in interpersonal and legal contexts ought to take appropriate actions to foster trust, provide care, and maintain social harmony when in situations that require attention to the well-being of others or the integrity of social interactions.
  [Middlemarch] n=743: A person in a specific role or relationship must perform a specific action or convey information in a specific con

## 2. Semantic Coherence Audit
For a random sample of merged clusters, show the canonical norm and all source norms side-by-side. Ask: do these belong together?

In [14]:
def show_cluster(row, max_sources=8):
    """Pretty-print a merged cluster for manual inspection."""
    sources = json.loads(row['source_norm_articulations'])
    amap = json.loads(row['abstraction_map']) if pd.notna(row['abstraction_map']) else {}
    
    print(f'━━━ [{row["title"]}] cluster_id={row["cluster_id"]}, n={row["cluster_size"]}, force={row["canonical_normative_force"]} ━━━')
    print(f'CANONICAL: {row["canonical_norm_articulation"]}')
    print(f'  subject: {row["canonical_norm_subject"]}')
    print(f'  act:     {row["canonical_norm_act"]}')
    print(f'  cond:    {row["canonical_condition_of_application"]}')
    print(f'  context: {row["canonical_context"]}')
    print(f'RATIONALE: {row["consolidation_rationale"]}')
    if amap:
        print(f'ABSTRACTION MAP:')
        for k, v in amap.items():
            if isinstance(v, dict):
                print(f'  {k}: {v.get("canonical","?")} ← {v.get("originals",[])}')  
    print(f'SOURCE NORMS ({len(sources)}):')
    for i, s in enumerate(sources[:max_sources]):
        print(f'  {i+1}. {s}')
    if len(sources) > max_sources:
        print(f'  ... +{len(sources) - max_sources} more')
    print()

In [15]:
# Sample 3 small clusters (3-5 members) and 3 medium clusters (6-15) per book for 1984 and P&P
for gid, title in [('1984', '1984'), ('1342', 'Pride & Prejudice')]:
    book = multi[multi['group_key'] == gid]
    print(f'\n{"="*80}')
    print(f'  {title}: {len(book)} merged clusters')
    print(f'{"="*80}\n')
    
    small = book[(book['cluster_size'] >= 3) & (book['cluster_size'] <= 5)]
    medium = book[(book['cluster_size'] >= 6) & (book['cluster_size'] <= 15)]
    
    print(f'--- Small clusters (3-5 members), sample of {min(3, len(small))} ---\n')
    for _, r in small.sample(n=min(3, len(small)), random_state=42).iterrows():
        show_cluster(r)
    
    print(f'--- Medium clusters (6-15 members), sample of {min(3, len(medium))} ---\n')
    for _, r in medium.sample(n=min(3, len(medium)), random_state=42).iterrows():
        show_cluster(r)


  1984: 18 merged clusters

--- Small clusters (3-5 members), sample of 3 ---

━━━ [1984] cluster_id=713, n=3, force=recommended ━━━
CANONICAL: A person in a survival situation under authoritarian rule ought to take actions to ensure personal safety and avoid detection.
  subject: a person in a survival situation under authoritarian rule
  act:     take actions to ensure personal safety and avoid detection
  cond:    when in a survival situation under authoritarian rule
  context: survival and resistance
RATIONALE: These norms were merged because they all recommend actions to ensure personal safety and avoid detection in a survival situation under authoritarian rule. The specific actions and conditions were abstracted to a more general form to cover all scenarios without losing the core prescriptive content.
ABSTRACTION MAP:
  norm_subject: a person in a survival situation under authoritarian rule ← ['a Party member', 'you (the individual)', 'one who is in hiding from the Thought Poli

## 3. Mega-Cluster Deep Dive
The largest clusters (>20 members) are most likely to be bad merges — semantically broad catch-alls.

In [16]:
mega = multi[multi['cluster_size'] > 20].sort_values('cluster_size', ascending=False)
print(f'{len(mega)} mega-clusters (>20 members)\n')
for _, r in mega.head(5).iterrows():
    show_cluster(r, max_sources=10)

24 mega-clusters (>20 members)

━━━ [Middlemarch] cluster_id=677, n=807, force=recommended ━━━
CANONICAL: Individuals should engage in actions that align with their roles and responsibilities when they have the ability or opportunity to do so.
  subject: individuals
  act:     engage in actions that align with their roles and responsibilities
  cond:    when they have the ability or opportunity to do so
  context: personal and interpersonal ethics
RATIONALE: These norms were merged because they all recommend actions that individuals should take based on their roles and responsibilities. The canonical norm abstracts the specific actions and conditions to a general principle that captures the shared prescriptive core. While some nuance in the specific actions and conditions is lost, the core recommendation remains intact.
ABSTRACTION MAP:
  norm_subject: individuals ← ['Celia', 'Dorothea', 'a landowner', 'a person', 'an individual in a position to support someone in significant endeavors

## 4. Per-Book Compression Analysis
How aggressively is each book compressed? High compression may indicate over-merging.

In [17]:
per_book = []
for gid in sorted(ID_TO_TITLE.keys()):
    n_raw = len(raw_norms[raw_norms['gutenberg_id'] == gid])
    book_consol = consolidated[consolidated['group_key'] == gid]
    n_consol = len(book_consol)
    n_multi = len(book_consol[book_consol['cluster_size'] > 1])
    n_singleton = len(book_consol[book_consol['cluster_size'] == 1])
    max_cluster = book_consol['cluster_size'].max()
    mean_cluster = book_consol[book_consol['cluster_size'] > 1]['cluster_size'].mean() if n_multi > 0 else 0
    compression = 1 - (n_consol / n_raw) if n_raw > 0 else 0
    
    per_book.append({
        'title': ID_TO_TITLE[gid],
        'raw': n_raw,
        'canonical': n_consol,
        'compression': f'{compression:.1%}',
        'merged_clusters': n_multi,
        'singletons': n_singleton,
        'max_cluster': max_cluster,
        'mean_cluster': f'{mean_cluster:.1f}',
    })

pb_df = pd.DataFrame(per_book)
display(pb_df)
print()
print('Books with >50% compression may warrant manual review:')
for r in per_book:
    pct = float(r['compression'].rstrip('%'))
    if pct > 50:
        print(f'  ⚠ {r["title"]}: {r["compression"]} compression ({r["raw"]} → {r["canonical"]})')

,title,raw,canonical,compression,merged_clusters,singletons,max_cluster,mean_cluster
0,Bleak House,2392,1448,39.5%,112,1336,296,9.3
1,Alice,194,100,48.5%,11,89,32,9.5
2,Count of Monte Cristo,2976,1457,51.0%,127,1330,805,12.8
3,Pride & Prejudice,967,572,40.8%,48,524,194,9.2
4,Les Mis,3746,2598,30.6%,267,2331,39,5.2
5,Anna Karenina,1867,1333,28.6%,112,1221,23,5.7
6,Middlemarch,2291,438,80.9%,22,416,807,85.1
7,1984,742,182,75.5%,18,164,376,31.7
8,Dorian Gray,325,175,46.2%,15,160,78,10.7
9,Age of Innocence,476,231,51.5%,13,218,149,19.1



Books with >50% compression may warrant manual review:
  ⚠ Count of Monte Cristo: 51.0% compression (2976 → 1457)
  ⚠ Middlemarch: 80.9% compression (2291 → 438)
  ⚠ 1984: 75.5% compression (742 → 182)
  ⚠ Age of Innocence: 51.5% compression (476 → 231)


## 5. Information Flow Norms
How many consolidated norms govern information flow? These are the most CI-relevant.

In [18]:
info_flow = consolidated[consolidated['canonical_governs_info_flow'] == True]
non_info = consolidated[consolidated['canonical_governs_info_flow'] != True]

print(f'Governs information flow: {len(info_flow):,} ({len(info_flow)/len(consolidated)*100:.1f}%)')
print(f'Non-information-flow:     {len(non_info):,} ({len(non_info)/len(consolidated)*100:.1f}%)')
print()

print('Per-book info-flow norms:')
for gid in sorted(ID_TO_TITLE.keys()):
    book = consolidated[consolidated['group_key'] == gid]
    n_if = len(book[book['canonical_governs_info_flow'] == True])
    print(f'  {ID_TO_TITLE[gid]:25s}: {n_if:4d} / {len(book):4d} ({n_if/len(book)*100:.1f}%)')

print()
print('Sample info-flow norms:')
for _, r in info_flow.sample(n=min(5, len(info_flow)), random_state=42).iterrows():
    print(f'  [{r["title"]}] {r["canonical_norm_articulation"]}')
    if pd.notna(r['canonical_info_flow_note']):
        print(f'    CI note: {r["canonical_info_flow_note"]}')

Governs information flow: 976 (11.4%)
Non-information-flow:     7,558 (88.6%)

Per-book info-flow norms:
  Bleak House              :  217 / 1448 (15.0%)
  Alice                    :   11 /  100 (11.0%)
  Count of Monte Cristo    :  231 / 1457 (15.9%)
  Pride & Prejudice        :   78 /  572 (13.6%)
  Les Mis                  :  207 / 2598 (8.0%)
  Anna Karenina            :  120 / 1333 (9.0%)
  Middlemarch              :   43 /  438 (9.8%)
  1984                     :   29 /  182 (15.9%)
  Dorian Gray              :   14 /  175 (8.0%)
  Age of Innocence         :   26 /  231 (11.3%)

Sample info-flow norms:
  [Bleak House] My dear (Esther Summerson) should give an opinion when asked by Mrs. Woodcourt.
    CI note: sender=Esther Summerson, recipient=Mrs. Woodcourt, information_type=opinion, transmission_principle=honesty
  [Anna Karenina] The rival department should be required to provide evidence of measures taken to account for actions taken over the past ten years.
    CI note: send

## 6. Spot-Check: Canonical vs Source Semantic Distance
Embed canonical articulations and their source norms, compute cosine similarity to flag potential semantic drift.

In [19]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

# Compute similarity for all merged clusters
sims = []
for _, row in multi.iterrows():
    canonical = str(row['canonical_norm_articulation'])
    sources = json.loads(row['source_norm_articulations'])
    if not sources or not canonical:
        continue
    
    all_texts = [canonical] + sources
    embs = model.encode(all_texts, normalize_embeddings=True)
    # Cosine similarity between canonical and each source
    canon_emb = embs[0:1]
    source_embs = embs[1:]
    cos_sims = np.dot(source_embs, canon_emb.T).flatten()
    
    sims.append({
        'group_key': row['group_key'],
        'title': row['title'],
        'cluster_id': row['cluster_id'],
        'cluster_size': row['cluster_size'],
        'mean_sim': cos_sims.mean(),
        'min_sim': cos_sims.min(),
        'max_sim': cos_sims.max(),
        'canonical': canonical[:100],
    })

sim_df = pd.DataFrame(sims)
print(f'Canonical-to-source similarity stats (n={len(sim_df)} clusters):')
print(sim_df[['mean_sim', 'min_sim', 'max_sim']].describe())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Canonical-to-source similarity stats (n=745 clusters):
         mean_sim     min_sim     max_sim
count  745.000000  745.000000  745.000000
mean     0.649822    0.500375    0.799295
std      0.172702    0.221570    0.152037
min      0.147738   -0.039024    0.211092
25%      0.549371    0.333985    0.730020
50%      0.674978    0.515576    0.840755
75%      0.781311    0.676896    0.905640
max      1.000000    1.000000    1.000000


In [20]:
# Flag clusters where canonical drifted far from sources
low_sim = sim_df[sim_df['mean_sim'] < 0.5].sort_values('mean_sim')
print(f'\nClusters with mean similarity < 0.5 (potential bad merges): {len(low_sim)}')
for _, r in low_sim.head(10).iterrows():
    print(f'  [{r["title"]}] sim={r["mean_sim"]:.3f} n={r["cluster_size"]}: {r["canonical"]}')


Clusters with mean similarity < 0.5 (potential bad merges): 144
  [Les Mis] sim=0.148 n=8: The individual may perform specific actions under certain conditions.
  [Count of Monte Cristo] sim=0.152 n=108: A person may perform an action under specific conditions.
  [Pride & Prejudice] sim=0.159 n=17: The individual may perform a specific action under certain circumstances.
  [Les Mis] sim=0.160 n=7: The individual must perform tasks or transactions when in a relevant context and under specific cond
  [Les Mis] sim=0.163 n=3: The individual may engage in a transaction for a specified amount if the other party agrees to the t
  [Middlemarch] sim=0.169 n=43: A person may perform an action when the person is in a position to do so and the action is appropria
  [Pride & Prejudice] sim=0.176 n=194: Individuals in a position of responsibility must fulfill their duties and obligations when in a posi
  [Anna Karenina] sim=0.196 n=21: The individual may perform an action when the context or situa

In [21]:
# Show the worst offenders in detail
if len(low_sim) > 0:
    print('\n=== Detailed view of lowest-similarity clusters ===\n')
    worst_ids = low_sim.head(3)[['group_key', 'cluster_id']].values
    for gk, cid in worst_ids:
        row = multi[(multi['group_key'] == gk) & (multi['cluster_id'] == cid)].iloc[0]
        show_cluster(row)
else:
    print('All clusters have mean similarity >= 0.5 — no obvious bad merges.')


=== Detailed view of lowest-similarity clusters ===

━━━ [Les Mis] cluster_id=559, n=8, force=permitted ━━━
CANONICAL: The individual may perform specific actions under certain conditions.
  subject: the individual
  act:     perform specific actions
  cond:    under certain conditions
  context: legal/judicial
RATIONALE: These norms were merged because they all permit specific actions under certain conditions. The canonical norm abstracts the specific individuals and actions to a general form, capturing the shared prescriptive core. The nuance of the specific actions and conditions is lost in this abstraction, but the overall permissive nature is preserved.
ABSTRACTION MAP:
  norm_subject: the individual ← ['Jean Valjean', 'the gendarmes', 'Madeleine', 'Javert', 'the speaker', 'Cosette']
  norm_act: perform specific actions ← ['take the candlesticks', 'release Jean Valjean', "authorize Fantine's release", 'allow Bigrenaille to have tobacco while in confinement', 'go home for a short 

## 7. Context Distribution
What societal contexts are represented in the normative universes?

In [22]:
print('Top 20 contexts across all consolidated norms:')
ctx_counts = consolidated['canonical_context'].value_counts().head(20)
for ctx, n in ctx_counts.items():
    print(f'  {ctx:40s}: {n:4d}')
print()

# Per-book context diversity
print('Context diversity per book:')
for gid in sorted(ID_TO_TITLE.keys()):
    book = consolidated[consolidated['group_key'] == gid]
    n_unique = book['canonical_context'].nunique()
    top_ctx = book['canonical_context'].value_counts().head(3).index.tolist()
    print(f'  {ID_TO_TITLE[gid]:25s}: {n_unique:3d} unique contexts, top: {", ".join(top_ctx)}')

Top 20 contexts across all consolidated norms:
  interpersonal ethics                    : 3864
  legal/judicial                          :  643
  family                                  :  603
  governance                              :  448
  commerce                                :  233
  communal life                           :  198
  divine-human relationship               :  161
  education/transmission                  :  121
  worship/ritual                          :   92
  personal conduct                        :   78
  household management                    :   66
  personal development                    :   54
  hospitality                             :   42
  social etiquette                        :   39
  interpersonal duties                    :   33
  personal ethics                         :   26
  ritual                                  :   25
  professional ethics                     :   25
  military                                :   25
  healthcare          